# 01 Cloudless DET Feedback GA Search

Generated notebook for JOILang GA feedback experiments.

> 실행 전 `BASE_DIR`, `MODEL_KEY`, `DEVICE`, API key/env를 확인하세요.

## 0. 목적과 실험 흐름

이 노트북은 **cloudless Strict DET feedback만으로 GA prompt search가 얼마나 개선되는지**를 단계적으로 검증한다.

실험 순서:
1. row 1개 smoke
2. 해당 row의 DET feedback / candidate / block diff 분석
3. manual feedback 또는 prompt rule 추가 후 같은 row 재실행
4. category 단위 반복
5. 전체 280개, 10 generation 장시간 실행
6. generation별 DETPass, AvgDET, token, Pareto 그래프 생성

In [ ]:
import os
import sys
import json
import csv
import shlex
import time
import subprocess
from pathlib import Path
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 240)

BASE_DIR = Path(os.environ.get("JOILANG_BASE_DIR", "/root/llm/JOILang-Server")).expanduser().resolve()
VERSION_ROOT = BASE_DIR / "gpt_mg" / "version0_15_update20260413"
GA_SCRIPT = VERSION_ROOT / "scripts" / "run_ga_search.py"
RUN_BENCHMARK = VERSION_ROOT / "scripts" / "run_benchmark.py"
EVAL_PIPELINE = BASE_DIR / "run_eval_pipeline_check.sh"

DATASET = BASE_DIR / "datasets" / "JOICommands-280.csv"
SERVICE_SCHEMA = BASE_DIR / "datasets" / "service_list_ver2.0.1.json"
DEFAULT_GENOME = VERSION_ROOT / "genomes" / "example_genome.json"

PYTHON = os.environ.get("JOI_V15_PYTHON", sys.executable)
WORKER_PYTHON = os.environ.get("JOI_V15_WORKER_PYTHON", PYTHON)

MODEL_KEY = os.environ.get("MODEL_KEY", "qwen25_coder_14b")
DEVICE = os.environ.get("JOI_V15_LOCAL_DEVICE", "cuda:0")
LOCAL_MODELS_BASE = Path(os.environ.get("JOI_V15_LOCAL_MODEL_BASE_DIR", str(BASE_DIR.parent / "local_models"))).expanduser()
LOCAL_MODEL_DIR = Path(os.environ.get("JOI_V15_LOCAL_MODEL_NAME", str(LOCAL_MODELS_BASE / MODEL_KEY))).expanduser()

RUN_TAG = os.environ.get("RUN_TAG", datetime.now().strftime("%Y%m%d_%H%M%S"))
NOTEBOOK_RUN_ROOT = BASE_DIR / "artifacts" / "notebook_ga_runs" / RUN_TAG
NOTEBOOK_RUN_ROOT.mkdir(parents=True, exist_ok=True)

ENV = os.environ.copy()
ENV.update({
    "JOI_V15_PYTHON": PYTHON,
    "JOI_V15_WORKER_PYTHON": WORKER_PYTHON,
    "JOI_V15_LOCAL_MODEL_BASE_DIR": str(LOCAL_MODELS_BASE),
    "JOI_V15_LOCAL_MODEL_NAME": str(LOCAL_MODEL_DIR),
    "JOI_V15_LOCAL_FILES_ONLY": os.environ.get("JOI_V15_LOCAL_FILES_ONLY", "true"),
    "JOI_V15_LOCAL_DEVICE": DEVICE,
    "JOI_V15_LOCAL_DTYPE": os.environ.get("JOI_V15_LOCAL_DTYPE", "bf16"),
    "JOI_V15_LOCAL_LOAD_IN_4BIT": os.environ.get("JOI_V15_LOCAL_LOAD_IN_4BIT", "false"),
    "JOI_V15_LOCAL_TRUST_REMOTE_CODE": os.environ.get("JOI_V15_LOCAL_TRUST_REMOTE_CODE", "true"),
    "TRANSFORMERS_VERBOSITY": "error",
    "HF_HUB_DISABLE_PROGRESS_BARS": "1",
    "TOKENIZERS_PARALLELISM": "false",
    "PYTHONFAULTHANDLER": "1",
})

print("BASE_DIR:", BASE_DIR)
print("VERSION_ROOT:", VERSION_ROOT)
print("GA_SCRIPT:", GA_SCRIPT, GA_SCRIPT.exists())
print("DATASET:", DATASET, DATASET.exists())
print("SERVICE_SCHEMA:", SERVICE_SCHEMA, SERVICE_SCHEMA.exists())
print("MODEL_KEY:", MODEL_KEY)
print("LOCAL_MODEL_DIR:", LOCAL_MODEL_DIR, LOCAL_MODEL_DIR.exists())
print("DEVICE:", DEVICE)
print("NOTEBOOK_RUN_ROOT:", NOTEBOOK_RUN_ROOT)

assert BASE_DIR.exists(), BASE_DIR
assert GA_SCRIPT.exists(), GA_SCRIPT
assert DATASET.exists(), DATASET
assert SERVICE_SCHEMA.exists(), SERVICE_SCHEMA

def ts():
    return datetime.now().strftime("%Y%m%d_%H%M%S")

def run_cmd(cmd, *, cwd=BASE_DIR, env=ENV, log_path=None, check=True):
    """Run a shell command list, stream output, and optionally tee to a log file."""
    cmd = [str(x) for x in cmd]
    print("\n[CMD]")
    print(" ".join(shlex.quote(x) for x in cmd))
    if log_path is not None:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        print("[LOG]", log_path)
    proc = subprocess.Popen(
        cmd, cwd=str(cwd), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    lines = []
    with (open(log_path, "w", encoding="utf-8") if log_path else open(os.devnull, "w", encoding="utf-8")) as lf:
        for line in proc.stdout:
            print(line, end="")
            lines.append(line)
            if log_path:
                lf.write(line)
    rc = proc.wait()
    if check and rc != 0:
        raise RuntimeError(f"command failed rc={rc}: {' '.join(cmd)}")
    return rc, "".join(lines)

def ga_common_args():
    return [
        PYTHON, "-u", str(GA_SCRIPT),
        "--profile", "version0_15",
        "--genome-json", str(DEFAULT_GENOME),
        "--dataset", str(DATASET),
        "--service-schema", str(SERVICE_SCHEMA),
        "--model-key", MODEL_KEY,
        "--llm-mode", "worker",
        "--candidate-k", "1",
        "--repair-attempts", "0",
        "--det-profile", "strict",
        "--selection-mode", "redesign",
        "--fitness-mode", "phase_aware",
        "--mutation-mode", "cloudless_decompiler",
        "--category-balance-mode", "guard",
        "--token-penalty-mode", "hybrid",
        "--stop-controller-mode", "active",
        "--reasoning-mutation-mode", "auto",
        "--intent-hint-mode", "auto",
        "--feedback-guided-mutation",
        "--enable-compression-mutation",
        "--enable-prompt-decompiler",
        "--enable-rendered-prompt-dedupe",
        "--enable-pareto-archive",
        "--enable-group-specialist-archives",
        "--full-run",
        "--force",
        "--progress", "verbose",
        "--retries", "0",
        "--target-detpass", "90",
    ]

def run_ga(label, scope_args, tuning_args=None, extra_args=None, output_root=None, check=True):
    output_root = Path(output_root or (NOTEBOOK_RUN_ROOT / label)).resolve()
    output_root.mkdir(parents=True, exist_ok=True)
    log_path = output_root / f"{label}.log"
    cmd = ga_common_args()
    cmd += list(scope_args)
    cmd += list(tuning_args or [])
    cmd += ["--output-root", str(output_root)]
    cmd += list(extra_args or [])
    rc, output = run_cmd(cmd, log_path=log_path, check=check)
    return output_root

def load_json(path):
    path = Path(path)
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))

def read_csv_if_exists(path):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path)

def latest_file(root, pattern):
    files = sorted(Path(root).glob(pattern), key=lambda p: p.stat().st_mtime)
    return files[-1] if files else None

def summarize_ga_run(run_dir):
    run_dir = Path(run_dir)
    summary = load_json(run_dir / "ga_summary.json")
    best = load_json(run_dir / "best_genome.json")
    print("RUN_DIR:", run_dir)
    print("best_DETPass:", summary.get("best_DETPass") or summary.get("accepted_best_DETPass"))
    print("best_avg_DET:", summary.get("best_avg_DET") or summary.get("accepted_best_avg_DET"))
    print("best_genome_id:", best.get("id") or best.get("genome_id"))
    print("stop_reason:", summary.get("stop_reason"))
    for name in ["ga_summary.json", "best_genome.json", "ga_block_diffs.jsonl", "advisor_mutation_summary.csv"]:
        p = run_dir / name
        print(f"{name}:", p.exists(), p)
    return summary, best

def collect_candidate_tables(run_dir):
    cand_dir = Path(run_dir) / "candidates"
    dfs = []
    for p in sorted(cand_dir.glob("*.csv")):
        try:
            df = pd.read_csv(p)
            df["source_file"] = str(p)
            dfs.append(df)
        except Exception as e:
            print("failed:", p, e)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

def collect_run_table(run_dirs):
    rows = []
    for rd in map(Path, run_dirs):
        s = load_json(rd / "ga_summary.json")
        b = load_json(rd / "best_genome.json")
        rows.append({
            "run_dir": str(rd),
            "label": rd.name,
            "best_DETPass": s.get("best_DETPass") or s.get("accepted_best_DETPass"),
            "best_avg_DET": s.get("best_avg_DET") or s.get("accepted_best_avg_DET"),
            "best_prompt_tokens": s.get("best_avg_prompt_tokens") or s.get("accepted_best_avg_prompt_tokens"),
            "stop_reason": s.get("stop_reason"),
            "best_genome_id": b.get("id") or b.get("genome_id"),
        })
    return pd.DataFrame(rows)

def generation_history(run_dir):
    s = load_json(Path(run_dir) / "ga_summary.json")
    hist = s.get("best_history") or s.get("generation_history") or []
    if not hist:
        return pd.DataFrame()
    df = pd.DataFrame(hist)
    df["run_dir"] = str(run_dir)
    return df

def plot_generation_history(run_dirs):
    hdfs = [generation_history(rd) for rd in run_dirs]
    hdfs = [df for df in hdfs if not df.empty]
    if not hdfs:
        print("No generation history found.")
        return pd.DataFrame()
    hist = pd.concat(hdfs, ignore_index=True)
    display(hist.head())

    gen_col = "generation" if "generation" in hist.columns else hist.columns[0]
    det_col = "train_det_pass_rate" if "train_det_pass_rate" in hist.columns else ("DETPass" if "DETPass" in hist.columns else None)
    avg_col = "avg_det_score" if "avg_det_score" in hist.columns else ("avg_DET" if "avg_DET" in hist.columns else None)
    tok_col = "avg_prompt_tokens" if "avg_prompt_tokens" in hist.columns else None

    if det_col:
        plt.figure(figsize=(8, 4))
        for rd, g in hist.groupby("run_dir"):
            plt.plot(g[gen_col], g[det_col], marker="o", label=Path(rd).name)
        plt.xlabel("Generation")
        plt.ylabel("DETPass / pass rate")
        plt.title("GA DETPass by generation")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

    if avg_col:
        plt.figure(figsize=(8, 4))
        for rd, g in hist.groupby("run_dir"):
            plt.plot(g[gen_col], g[avg_col], marker="o", label=Path(rd).name)
        plt.xlabel("Generation")
        plt.ylabel("Average DET")
        plt.title("GA average DET by generation")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

    if tok_col:
        plt.figure(figsize=(8, 4))
        for rd, g in hist.groupby("run_dir"):
            plt.plot(g[gen_col], g[tok_col], marker="o", label=Path(rd).name)
        plt.xlabel("Generation")
        plt.ylabel("Avg prompt tokens")
        plt.title("Prompt-token compression by generation")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

    return hist

def inspect_failures(run_dir, max_rows=30):
    df = collect_candidate_tables(run_dir)
    if df.empty:
        print("No candidate CSV rows found.")
        return df
    cols = [c for c in df.columns if any(k in c.lower() for k in ["row", "category", "det", "pass", "failure", "error", "prompt", "token", "candidate", "genome"])]
    display(df[cols].head(max_rows))
    return df

def compare_two_runs(run_a, run_b):
    a = collect_candidate_tables(run_a)
    b = collect_candidate_tables(run_b)
    if a.empty or b.empty:
        print("candidate table missing")
        return pd.DataFrame()
    # Use best effort join keys
    key_candidates = ["row_no", "row_id", "dataset_row", "index"]
    key = next((k for k in key_candidates if k in a.columns and k in b.columns), None)
    if key is None:
        print("No common row key. Showing summaries only.")
        display(pd.DataFrame([summarize_ga_run(run_a)[0], summarize_ga_run(run_b)[0]]))
        return pd.DataFrame()
    pass_cols = [c for c in a.columns if "pass" in c.lower() or "det" in c.lower()]
    a_small = a[[key] + pass_cols].copy()
    b_small = b[[key] + [c for c in b.columns if "pass" in c.lower() or "det" in c.lower()]].copy()
    merged = a_small.merge(b_small, on=key, suffixes=("_a", "_b"))
    display(merged.head(50))
    return merged

## 1. Row 1개 smoke: cloudless DET feedback GA

In [ ]:
ROW_NO = 1
ROW_TUNING = [
    "--population", "4",
    "--gens", "2",
    "--min-generations", "2",
    "--max-generations", "2",
    "--sample-size", "1",
    "--validation-size", "1",
    "--cheap-eval-limit", "1",
    "--plateau-window", "1",
    "--disruptive-max-attempts", "1",
    "--timeout-sec", "2400",
]
row1_run = run_ga(
    label=f"cloudless_row{ROW_NO:03d}_g2_{ts()}",
    scope_args=["--start-row", str(ROW_NO), "--end-row", str(ROW_NO)],
    tuning_args=ROW_TUNING,
)
summary, best = summarize_ga_run(row1_run)

## 2. Row feedback 상세 분석

In [ ]:
row1_candidates = inspect_failures(row1_run, max_rows=50)
block_diffs = Path(row1_run) / "ga_block_diffs.jsonl"
if block_diffs.exists():
    print(block_diffs)
    for i, line in enumerate(block_diffs.read_text(encoding="utf-8").splitlines()[:20], start=1):
        print(f"\n--- diff {i} ---")
        print(line[:2000])
else:
    print("No ga_block_diffs.jsonl")

## 3. Manual feedback 작성 후 같은 row 재실행

아래 `MANUAL_FEEDBACK`에 row 분석 결과를 바탕으로 prompt mutation 방향을 직접 적는다.
`run_feedback_loop.py` 계열은 `VERSION_ROOT/logs/**/manual_feedback.md` 류의 파일을 읽도록 설계되어 있으므로, 같은 convention으로 기록한다.

In [ ]:
MANUAL_FEEDBACK = """
# Manual feedback for row-level GA rerun

- Preserve canonical service/function names exactly.
- Prefer the minimal JOILang program that directly satisfies the command.
- Do not add unrelated guards, devices, or helper actions.
""".strip()

feedback_dir = VERSION_ROOT / "logs" / "notebook_manual_feedback" / f"row{ROW_NO:03d}_{ts()}"
feedback_dir.mkdir(parents=True, exist_ok=True)
(feedback_dir / "manual_feedback.md").write_text(MANUAL_FEEDBACK + "\n", encoding="utf-8")
print("Wrote:", feedback_dir / "manual_feedback.md")

row1_rerun = run_ga(
    label=f"cloudless_row{ROW_NO:03d}_manual_rerun_g2_{ts()}",
    scope_args=["--start-row", str(ROW_NO), "--end-row", str(ROW_NO)],
    tuning_args=ROW_TUNING,
)
summary2, best2 = summarize_ga_run(row1_rerun)
compare_two_runs(row1_run, row1_rerun)

## 4. Category 단위 실험

In [ ]:
RUN_CATEGORY_SWEEP = False  # 실행하려면 True
CATEGORY_LIMIT_PER_CATEGORY = 5
CATEGORY_TUNING = [
    "--population", "6",
    "--gens", "3",
    "--min-generations", "2",
    "--max-generations", "3",
    "--sample-size", "4",
    "--validation-size", "4",
    "--cheap-eval-limit", "2",
    "--plateau-window", "1",
    "--disruptive-max-attempts", "1",
    "--timeout-sec", "3600",
    "--limit-per-category", str(CATEGORY_LIMIT_PER_CATEGORY),
]

category_runs = []
if RUN_CATEGORY_SWEEP:
    for cat in range(1, 9):
        rd = run_ga(
            label=f"cloudless_category{cat}_g3_{ts()}",
            scope_args=["--category", str(cat)],
            tuning_args=CATEGORY_TUNING,
        )
        category_runs.append(rd)
        summarize_ga_run(rd)

if category_runs:
    display(collect_run_table(category_runs))
    plot_generation_history(category_runs)

## 5. 전체 280개, 10 generation 실행

In [ ]:
RUN_FULL_10GEN = False  # 장시간 실행 전 True로 변경

FULL_TUNING = [
    "--population", "16",
    "--gens", "10",
    "--min-generations", "5",
    "--max-generations", "10",
    "--sample-size", "40",
    "--validation-size", "40",
    "--cheap-eval-limit", "20",
    "--plateau-window", "3",
    "--disruptive-max-attempts", "3",
    "--timeout-sec", "7200",
]

full_run = None
if RUN_FULL_10GEN:
    full_run = run_ga(
        label=f"cloudless_full280_g10_{ts()}",
        scope_args=["--category", "1", "--category", "2", "--category", "3", "--category", "4",
                    "--category", "5", "--category", "6", "--category", "7", "--category", "8"],
        tuning_args=FULL_TUNING,
    )
    summarize_ga_run(full_run)

## 6. 최종 그래프 및 산출물 분석

In [ ]:
# 분석할 run들을 직접 추가 가능
analysis_runs = [p for p in [locals().get("row1_run"), locals().get("row1_rerun"), locals().get("full_run")] if p]
analysis_runs += category_runs if "category_runs" in globals() else []
analysis_runs = [Path(p) for p in analysis_runs if p]

if not analysis_runs:
    # 최근 notebook run root 아래의 GA run들을 자동 수집
    analysis_runs = sorted([p for p in NOTEBOOK_RUN_ROOT.glob("*") if (p / "ga_summary.json").exists()], key=lambda p: p.stat().st_mtime)

print("analysis_runs:")
for p in analysis_runs:
    print("-", p)

display(collect_run_table(analysis_runs))
hist = plot_generation_history(analysis_runs)

# Pareto CSV가 있으면 별도 그래프
for rd in analysis_runs:
    pareto = read_csv_if_exists(Path(rd) / "pareto_rows.csv")
    if not pareto.empty and {"det_pass_rate", "avg_prompt_tokens"}.issubset(pareto.columns):
        plt.figure(figsize=(6, 4))
        plt.scatter(pareto["avg_prompt_tokens"], pareto["det_pass_rate"])
        plt.xlabel("Avg prompt tokens")
        plt.ylabel("DET pass rate")
        plt.title(f"Pareto scatter: {Path(rd).name}")
        plt.grid(True, alpha=0.3)
        plt.show()